# Zarattini (2023) Opening-Bias ORB — Orbis Universe Validation

**Source strategy:** [giovannibrusco/zarattini-2023-orb-qqq](https://github.com/giovannibrusco/zarattini-2023-orb-qqq) (MIT) — replication of Zarattini & Aziz, SSRN 4416622.

**Universe:** Orbis **ORB liquid sleeve** (`pipeline.compute.opening_range.LIQUID`) plus **QQQ** benchmark — the same seed list the nightly ORB job scans.

**Data:** Yahoo Finance **5-minute** bars (max ~60 calendar days on free tier).

**Adaptation:** Paper uses **NQ @ 09:25** confirmation; here we use **QQQ @ 09:25** for the filtered scenario when trading single names.

Paper only — not investment advice.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pipeline.research.run_zarattini_orb_validation import run_validation
from pipeline.research.zarattini_orb_data import orb_universe_symbols

OUT = Path("notebooks/output")
OUT.mkdir(parents=True, exist_ok=True)

print("Universe size:", len(orb_universe_symbols()))
print("Sample:", orb_universe_symbols()[:8], "…")

In [ ]:
# Full liquid list (~31 names). Set max_symbols=12 for a quick smoke run.
summary = run_validation(period="60d", max_symbols=None, capital_per_symbol=25_000.0)
summary

In [ ]:
rows = []
for scen, block in summary["scenarios"].items():
    p = block["portfolio"]
    rows.append({
        "scenario": scen,
        "trades": p["trades"],
        "net_pnl": round(p["net_pnl"], 2),
        "final_equity": round(p["final_equity"], 2),
        "win_rate": round(p["win_rate"], 3) if p["win_rate"] == p["win_rate"] else None,
        "mean_pnl/share": round(p["mean_pnl_per_share"], 4) if p["mean_pnl_per_share"] == p["mean_pnl_per_share"] else None,
        "sharpe_daily": round(p["sharpe_daily"], 2) if p["sharpe_daily"] == p["sharpe_daily"] else None,
        "max_dd": round(p["max_drawdown"], 3),
        "stop_%": round(p.get("stop_pct", 0) * 100, 1),
    })
results_df = pd.DataFrame(rows)
results_df

In [ ]:
fig_path = OUT / "equity_curves_all_scenarios.png"
from IPython.display import Image, display
if fig_path.exists():
    display(Image(filename=str(fig_path)))
else:
    print("Run validation cell first.")

In [ ]:
for scen in summary["scenarios"]:
    p = OUT / f"equity_curve_{scen}.png"
    if p.exists():
        print(scen)
        display(Image(filename=str(p)))

In [ ]:
# Per-symbol leaderboard (slippage scenario)
scen = "slippage_2c"
per = summary["scenarios"][scen]["per_symbol"]
leader = pd.DataFrame(per).T.sort_values("net_pnl", ascending=False)
leader.head(15)